# สร้าง AI Agent ขายกาแฟ

เมื่อวานเรากดปุ่มเอง วันนี้ให้โปรแกรมทำแทน ทีละขั้น:

1. **agent ด้วยกฎ** — วนดูตลาด ตัดสินใจด้วย Python ส่งข้อเสนอเอง
2. **เรียก LLM ครั้งแรก** — ขอคำตอบเป็น JSON และรับมือเมื่อ LLM พลาด
3. **ให้ LLM ช่วยตัดสินใจ** — LLM เลือกตัวคูณราคา Python คุมตัวเลข
4. **starter agent** — agent ตั้งต้นที่ทีมจะเอาไปแก้ต่อจนถึงวันแข่ง

**วิธีใช้:** รันทีละช่อง (Shift+Enter) จากบนลงล่าง · ช่องที่วนเล่นตลาดหยุดได้ด้วยปุ่ม ■ (Interrupt)

ถ้าขึ้น `No module named ...` ให้รันใน Terminal: `pip install httpx openai`

## 0. ตั้งค่า

ใช้ **ตลาดฝึก** (sandbox) และ key ของทีม · LLM ใช้ gateway ของผู้จัด (key เดียวกับตลาด) · **ใส่ key ที่บรรทัด `KEY` ที่เดียว** `LLM_KEY` ไม่ต้องแก้
ฝึกนอกเวลาแข่งจะใช้ Ollama บนเครื่องตัวเองก็ได้: `LLM_URL = "http://localhost:11434/v1"`, `LLM_MODEL = "qwen3.5:4b"` — **ช่วงแข่ง 2–3 ใช้ได้เฉพาะ gateway ของผู้จัด** LLM อื่น (cloud หรือ Ollama) ผิดกติกา ตัดสิทธิ์

In [ ]:
import json
import time

import httpx
from openai import OpenAI

MARKET = "http://<ที่อยู่ตลาดฝึก>:8000"      # ทีมงานจะแจ้งในห้อง
KEY = "sk-T00-ใส่key-ของทีม"                 # key ของทีม ใส่ที่นี่ที่เดียว (ขึ้นต้นด้วย sk-T)

LLM_URL = "http://<ที่อยู่-LLM-gateway>:4000/v1"
LLM_KEY = KEY                                # ไม่ต้องแก้: gateway ใช้ key เดียวกับตลาด
LLM_MODEL = "event-llm"

# ตรวจก่อนว่าแก้ที่อยู่และ key แล้ว (ถ้าไม่ตรวจ จะได้ error ภาษาอังกฤษที่อ่านยาก)
if "<" in MARKET or "<" in LLM_URL:
    raise ValueError("ยังไม่ได้ใส่ที่อยู่: แก้ MARKET และ LLM_URL ในช่องนี้ตามที่ทีมงานแจ้ง")
if not KEY.isascii() or not KEY.startswith("sk-T") or KEY.startswith("sk-T00-"):
    raise ValueError("ยังไม่ได้ใส่ key ของทีม: แก้ KEY ในช่องนี้เป็น key ที่ทีมงานแจก (ขึ้นต้นด้วย sk-T ไม่มีภาษาไทย)")

api = httpx.Client(base_url=MARKET, headers={"Authorization": f"Bearer {KEY}"}, timeout=10)
llm = OpenAI(base_url=LLM_URL, api_key=LLM_KEY, timeout=60, max_retries=0)
print(api.get("/v1/version").json())

## 1. agent ด้วยกฎ

### 1.1 เครื่องมือจากเมื่อวาน: สูตร TV และการเช็กสเปก

วันนี้ผลวัดตัวอย่าง **คลาดเคลื่อนได้** — TV ที่คำนวณเป็นแค่ค่าประมาณ และล็อตที่ตัวอย่างผ่านสเปกพอดี ของจริงอาจไม่ผ่าน (`SPEC_MISMATCH`)

In [ ]:
# /v1/calendar ตอบสองส่วน: calendar (ปฏิทินตลาด) และ rules (ตัวเลขของสูตรมูลค่า กระสอบ คะแนน เวลา)
# ใช้แค่ rules · ดูคำตอบเต็มและความหมายทุกช่องได้ใน notebook เมื่อวาน (เรียกตลาดด้วย-Python ข้อ 4)
rules = api.get("/v1/calendar").json()["rules"]
BAG = rules["bag_kg"]                          # 1 กระสอบ = 60 กก.

def grade(d):
    """เกรดและตัวคูณขั้นเกรด จากข้อบกพร่อง d %"""
    for step in rules["defect_steps"]:
        if d <= step["max_defect_pct"]:
            return step["grade"], step["step"]

def true_value(sample):
    """TV = P0 × G × M × S จากผลวัดตัวอย่าง (วันนี้เป็นแค่ค่าประมาณ)"""
    d, m, s = sample["defect_pct"], sample["moisture_pct"], sample["screen"]
    mo = rules["moisture"]                    # M: ตัวคูณความชื้น
    if m <= mo["ref"]:
        mf = 1.0
    elif m <= mo["hard_limit"]:
        mf = (1 - m / 100) / (1 - mo["ref"] / 100) - mo["penalty_per_pp"] * (m - mo["ref"])
    else:
        mf = mo["hard_factor"]
    screens = {int(k): v for k, v in rules["screen_factors"].items()}
    sf = screens[min(max(s, min(screens)), max(screens))]
    return rules["p0"] * (1 - d / 100) * grade(d)[1] * mf * sf

def passes(sample, spec):
    """ผลวัดตัวอย่างผ่านสเปกของผู้ซื้อทุกข้อไหม"""
    return (sample["defect_pct"] <= spec["max_defect_pct"]
            and sample["moisture_pct"] <= spec["max_moisture_pct"]
            and (spec["min_screen"] is None or sample["screen"] >= spec["min_screen"]))

state = api.get("/v1/state").json()
for lot in state["lots"]:
    print(lot["lot_id"], lot["sample"], grade(lot["sample"]["defect_pct"])[0], round(true_value(lot["sample"]), 2))

### 1.2 สมองของ agent: ตัดสินใจหนึ่งรอบ

กฎง่ายๆ: ผู้ซื้อที่จ่ายดีได้ของก่อน · ล็อตมูลค่าสูงก่อน · ราคา = ตัวคูณของประเภทผู้ซื้อ × TV

ใหม่วันนี้: **จำกัดกระสอบต่อรอบ** — ทีมเสนอรวมได้ไม่เกิน `capacity_left_kg` ต่อรอบ จึงเทขายหมดในรอบเดียวไม่ได้

In [ ]:
MARKUP = {"SPECIALTY": 1.00, "COMMERCIAL": 1.00, "INSTANT": 1.00}   # ลองแก้แล้วเทียบผล

def decide_rules(state, markup):
    tv = {l["lot_id"]: true_value(l["sample"]) for l in state["lots"]}
    left = {l["lot_id"]: l["kg_available"] for l in state["lots"]}
    room = state["capacity_left_kg"] if state["capacity_left_kg"] is not None else 10**9
    offers = []
    for q in sorted(state["rfqs"], key=lambda q: -markup[q["buyer_type"]]):
        for lot in sorted(state["lots"], key=lambda l: -tv[l["lot_id"]]):
            if not passes(lot["sample"], q["spec"]):
                continue
            kg = min(left[lot["lot_id"]], q["kg"], room) // BAG * BAG
            if kg < BAG:
                continue
            offers.append({"rfq_id": q["rfq_id"], "lot_id": lot["lot_id"], "kg": kg,
                           "price_per_kg": round(markup[q["buyer_type"]] * tv[lot["lot_id"]], 2)})
            left[lot["lot_id"]] -= kg
            room -= kg
            break                     # RFQ ละ 1 ข้อเสนอ
    return offers

state = api.get("/v1/state").json()
print("ช่วง:", state["phase"], "· รอบนี้เสนอได้อีก", state["capacity_left_kg"], "กก.")
for o in decide_rules(state, MARKUP):
    print(o)

### 1.3 ลูปของ agent: ดู → ตัดสินใจ → ส่ง → อ่านผล

ทุก 1 วินาที อ่าน `GET /v1/state` แล้วทำตามช่วงของตลาด

- ช่วง `APPRAISAL`: ส่ง TV ทุกล็อตเป็น **ผลประเมิน** ให้เอง (`POST /v1/appraisals` ส่งได้ครั้งเดียวต่อเกม) · ถ้าเริ่มรันตอนเลยช่วงนี้ไปแล้ว ลูปจะเตือน

- `POST /v1/offers:batch` ส่งทุกข้อเสนอในครั้งเดียว · `replace_existing: true` = ยกเลิกของเดิมในรอบนี้ก่อน
- ถ้ามีข้อเสนอผิดแม้ตัวเดียว ทั้งชุดถูกปฏิเสธ (`BATCH_REJECTED`) พร้อมบอกว่าตัวไหนผิด
- อ่านเวลาจากตลาดเสมอ อย่าเขียนเวลารอบตายตัว (ตลาดฝึกรอบสั้นกว่าวันแข่ง)

In [ ]:
history = []          # ผลของเราแต่ละรอบ เป็นข้อความสั้นๆ ไว้ส่งให้ LLM ในข้อ 3

def show_results(res, tv):
    types = {r["rfq_id"]: r["buyer_type"] for r in res["market_summary"]}
    parts = []
    for o in res["my_offers"]:
        mk = o["price_per_kg"] / tv[o["lot_id"]] if o["lot_id"] in tv else 0
        parts.append(f"{types.get(o['rfq_id'], '?')} x{mk:.2f} {o['status']} {o['filled_kg'] // BAG}/{o['kg'] // BAG} bags")
    line = f"{res['round_id']}: " + ("; ".join(parts) or "no offers")
    history.append(line)
    print("   ผล", line)

def run(decide, max_rounds=None):
    acted, seen, warned = set(), set(), False
    while True:
        try:
            s = api.get("/v1/state").json()
        except httpx.HTTPError as e:              # เน็ตสะดุด: รอแล้วลองใหม่ อย่าให้ agent ตาย
            print("อ่านสถานะไม่ได้:", e)
            time.sleep(3)
            continue
        tv = {l["lot_id"]: true_value(l["sample"]) for l in s["lots"]}
        if not warned and s["phase"] in ("ROUND_OPEN", "ROUND_CLEARING", "CLOSED") and not s["appraisal_submitted"]:
            print("เกมนี้เลยช่วงประเมินแล้ว และยังไม่ได้ส่ง: คะแนนประเมินเป็น 0 · เริ่มเกมใหม่ด้วยช่องเริ่มเกมใหม่ข้างบน")
        warned = True
        if s["phase"] == "APPRAISAL" and not s["appraisal_submitted"]:
            items = [{"lot_id": i, "est_value_per_kg": round(v, 2)} for i, v in tv.items()]
            r = api.post("/v1/appraisals", json={"items": items})
            print("ส่งผลประเมิน", r.status_code)
        res = s.get("last_results")
        if res and res.get("round_id") and res["round_id"] not in seen:
            seen.add(res["round_id"])
            show_results(res, tv)
        if s["phase"] == "ROUND_OPEN" and s["round"]["round_id"] not in acted:
            if max_rounds and len(acted) >= max_rounds:
                print("ครบ", max_rounds, "รอบแล้ว หยุด")
                return
            acted.add(s["round"]["round_id"])
            t0 = time.time()
            offers = decide(s)
            r = api.post("/v1/offers:batch", json={"offers": offers, "replace_existing": True}) if offers else None
            print(f"{s['round']['round_id']} ({s['round']['kind']}) ส่ง {len(offers)} ข้อเสนอ "
                  f"{sum(o['kg'] for o in offers) // BAG} กระสอบ · คิด {time.time() - t0:.1f} วิ ·",
                  r.status_code if r else "-")
            if r is not None and r.status_code >= 400:
                print("   ", r.json()["error"])
        if s["phase"] == "CLOSED":
            print("ตลาดปิด:", s["standings"])
            return
        time.sleep(1)

**เริ่มเกมใหม่ก่อนรันลูป** (เหมือนข้อ 3 เมื่อวาน) จะได้ล็อตชุดใหม่ และลูปทันส่งผลประเมินตั้งแต่ต้นเกม · ใช้ช่องนี้ทุกครั้งที่อยากเริ่มเกมใหม่ ก่อนรันลูปในข้อ 1.3 หรือข้อ 3

ช่วงประเมินของตลาดฝึกวันนี้สั้น (ราว 1 นาที) รันช่องลูปข้างล่างต่อทันที

In [ ]:
r = api.post("/v1/sandbox/reset")      # เกมเดิมของทีมจบทันที ได้ล็อตชุดใหม่
s = r.json()
print(r.status_code, "ช่วง:", s["phase"], "· เหลือ", s["seconds_left"], "วินาที")

In [ ]:
run(lambda s: decide_rules(s, MARKUP), max_rounds=3)

**ลองเอง (5 นาที):** แก้ `MARKUP` ในข้อ 1.2 ให้แต่ละประเภทผู้ซื้อไม่เท่ากัน แล้วรันลูปอีกครั้ง — ผลเปลี่ยนอย่างไร

ดูสถานะ `ABOVE_LIMIT` (แพงเกินเพดาน) กับ `OUTBID` (แพ้ข้อเสนอที่คุ้มกว่า) เป็นตัวบอกทาง

## 2. เรียก LLM ครั้งแรก

LLM ของงานพูด API แบบเดียวกับ OpenAI (OpenAI-compatible) จึงใช้ไลบรารี `openai` ได้เลย
`reasoning_effort="none"` = ปิดโหมดคิดก่อนตอบ (thinking) — เร็วกว่ามาก

In [ ]:
t0 = time.time()
r = llm.chat.completions.create(
    model=LLM_MODEL, max_tokens=200, reasoning_effort="none",
    messages=[{"role": "user", "content": "อธิบายกาแฟโรบัสต้าใน 1 ประโยค"}])
print(r.choices[0].message.content)
print(f"ใช้เวลา {time.time() - t0:.1f} วิ · token ขาเข้า {r.usage.prompt_tokens} ขาออก {r.usage.completion_tokens}")

### 2.1 ขอคำตอบเป็น JSON ตาม schema

โปรแกรมอ่านคำตอบเป็นข้อความไม่ได้ จึงบอก LLM ว่าต้องตอบ JSON หน้าตาแบบไหน (`response_format` + JSON schema)

In [ ]:
SCHEMA = {"type": "object", "required": ["markup", "reason"], "properties": {
    "markup": {"type": "object", "required": ["SPECIALTY", "COMMERCIAL", "INSTANT"], "properties": {
        "SPECIALTY": {"type": "number"}, "COMMERCIAL": {"type": "number"}, "INSTANT": {"type": "number"}}},
    "reason": {"type": "string"}}}

r = llm.chat.completions.create(
    model=LLM_MODEL, max_tokens=300, reasoning_effort="none",
    messages=[{"role": "user", "content": "Suggest a price markup (x true value) for each buyer type of green coffee. "
                                          "Specialty pays the most, Instant the least."}],
    response_format={"type": "json_schema", "json_schema": {"name": "answer", "schema": SCHEMA}})
answer = json.loads(r.choices[0].message.content)
print(answer["markup"])
print(answer["reason"])

**สังเกต:** ได้ JSON ที่โปรแกรมอ่านได้ แต่ตัวเลขสมเหตุสมผลไหม? LLM ไม่รู้กติกาตลาดของเรา ถ้าไม่บอกข้อมูลก็เดาจากความรู้ทั่วไป
→ ต้อง (1) ให้ข้อมูลใน prompt ให้พอ และ (2) ให้ Python ตรวจตัวเลขก่อนใช้เสมอ

### 2.2 LLM พลาดได้เสมอ — agent ต้องไม่ตาย

LLM อาจช้า ล่ม ตอบไม่ใช่ JSON หรือ gateway ตอบ 429 (ทีมเรียกถี่เกิน) · ฟังก์ชันนี้ **ไม่ throw** — พลาดเมื่อไรคืน `None` แล้วให้กฎทำงานแทน

In [ ]:
def ask_json(system, user, schema, timeout=30):
    try:
        r = llm.with_options(timeout=timeout).chat.completions.create(
            model=LLM_MODEL, max_tokens=500, temperature=0.7, reasoning_effort="none",
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
            response_format={"type": "json_schema", "json_schema": {"name": "answer", "schema": schema}})
        return json.loads(r.choices[0].message.content)
    except Exception as e:              # timeout, 429, JSON เสีย ... อะไรก็ตาม
        print("LLM พลาด:", type(e).__name__, str(e)[:120])
        return None

print(ask_json("Answer JSON.", "Suggest markups.", SCHEMA, timeout=0.5))   # ตั้งใจให้ช้าเกินจนพลาด

## 3. ให้ LLM ช่วยตัดสินใจ

แบ่งงานให้ชัด: **LLM เลือกตัวคูณราคา** จากผลรอบก่อนๆ · **Python คุมตัวเลข** (ล็อต กระสอบ สเปก กรอบตัวคูณ)

ถ้า LLM ตอบไม่ทันหรือพลาด → ใช้ตัวคูณเดิมต่อ

In [ ]:
SYSTEM = """You set prices for a coffee farm in a sealed-offer market.
price = markup x the lot's true value. Buyers take the cheapest offers (price / true value) first
and reject any price above their hidden ceiling. Specialty usually pays the most, Instant the least.
ABOVE_LIMIT = too expensive for that buyer, OUTBID = another seller was cheaper, ACCEPTED = sold.
Unsold beans are worth only 0.80 x true value at the end.
Answer only JSON; keep the reason to one short sentence."""

markup = dict(MARKUP)
LOW, HIGH = 0.85, 1.30        # กรอบที่ Python ไม่ยอมให้ LLM ออกนอก

def decide_llm(state):
    user = "\n".join([
        f"Round {state['round']['round_id']} of {state['round']['of']}.",
        f"Current markups: {json.dumps(markup)}",
        "My recent results:", *(history[-5:] or ["none yet"]),
        "Choose the markups for this round."])
    answer = ask_json(SYSTEM, user, SCHEMA, timeout=max(3, (state["seconds_left"] or 30) - 10))
    if answer:
        for t in markup:
            try:
                markup[t] = round(min(max(float(answer["markup"][t]), LOW), HIGH), 3)
            except (KeyError, TypeError, ValueError):
                pass                   # ค่าไหนแปลกก็ใช้ค่าเดิม
        print("   LLM:", markup, "·", str(answer.get("reason"))[:120])
    else:
        print("   ใช้กฎ: ตัวคูณเดิม", markup)
    return decide_rules(state, markup)

run(decide_llm, max_rounds=3)

**คุยกัน:** LLM ปรับตัวคูณตามผลจริงไหม · ช้าแค่ไหน · ถ้า 55 ทีมเรียกพร้อมกันตอนรอบเปิดจะเป็นอย่างไร

## 4. starter agent

starter คือ agent ที่ทำทุกอย่างข้างบนครบ และรันจนตลาดปิดได้ทันที — ทีมเอาไปแก้ต่อ

เปิด Terminal ใน VS Code แล้วรัน (Windows):

```
cd starter
python -m venv .venv
.venv\Scripts\pip install -r requirements.txt
copy .env.example .env
```

แก้ `.env`: `MARKET_URL`, `TEAM_KEY`, `LLM_BASE_URL`, `LLM_MODEL`, `LLM_API_KEY` (ค่าเดียวกับข้อ 0) แล้วรัน

```
.venv\Scripts\python main.py --forever
```

`--forever` = จบเกมแล้วเล่นเกมใหม่ต่อเอง (ใช้กับตลาดฝึก) · หยุดด้วย Ctrl+C

ลำดับอ่านโค้ด: `main.py` → `strategy.decide()` → `guard()`

### 4.1 อ่าน log ของ starter

ทุกการตัดสินใจถูกบันทึกใน `starter/logs/decisions.jsonl` · `source` บอกว่ามาจาก `llm`, `llm+rules` หรือ `rules` (LLM พลาด)

In [ ]:
from pathlib import Path

LOG = Path("../../starter/logs/decisions.jsonl")    # แก้ path ให้ตรงกับที่เก็บ starter
if not LOG.exists():
    print("ยังไม่มี log: รัน starter ก่อน หรือแก้ path ให้ถูก")
rows = [json.loads(line) for line in LOG.read_text(encoding="utf-8").splitlines()] if LOG.exists() else []
for row in [r for r in rows if r["event"] == "offers_sent"][-5:]:
    print(row["time"][11:], row["round"], row["source"], row["n"], "ข้อเสนอ", row["kg"] // BAG, "กระสอบ",
          row["seconds"], "วิ")
    for note in row["notes"]:
        print("    ", note[:150])

### 4.2 แก้ 1 จุดแล้วเทียบผล

1. เปิด `starter/strategy.py` หา `MARKUP` แก้ค่าของประเภทผู้ซื้อหนึ่งประเภท
2. หยุด starter (Ctrl+C) แล้วรันใหม่ เล่นให้จบเกม
3. ดูมูลค่าจริงและ PR ของเกมที่จบแล้วด้วยช่องข้างล่าง เทียบกับเกมก่อนแก้

ผลแต่ละเกมแกว่งได้ เพราะล็อตและผู้ซื้อสุ่มใหม่ — เทียบหลายเกมก่อนสรุป

In [ ]:
r = api.get("/v1/sandbox/reveal")
if r.status_code != 200:
    print(r.json()["error"]["message"])          # ยังไม่จบเกม
else:
    rev = r.json()
    for lot in rev["lots"]:
        print(lot["lot_id"], "ตัวอย่าง", lot["sample"], "→ จริง", lot["true"], lot["true_grade"],
              "TV จริง", lot["true_value_per_kg"], "· เราประเมิน", lot["your_estimate"])
    sc = rev["score"]
    print(f"PR {sc['pr']} · คะแนนขาย {sc['pr_score']:.1f} + ประเมิน {sc['appraisal_score']:.1f} = {sc['total']:.1f} · อันดับ {sc['rank']}")

## ก่อนแข่งช่วงบ่าย

- บ่ายนี้ **ห้ามส่งคำสั่งด้วยมือ** — agent ต้องทำเองทั้งหมด หน้าเว็บดูได้อย่างเดียว
- ใช้ key ของวันแข่ง (คนละ key กับตลาดฝึก)
- ทดสอบ starter กับตลาดฝึกให้จบเกมได้ก่อน แล้วค่อยแก้ทีละจุด
- เรียก API ไม่เกิน 5 ครั้งต่อวินาที · เรียก LLM ไม่เกินที่ gateway กำหนด